# 3-11 apply自定义函数

`apply()` 可以把自定义函数应用到 Series 的元素，或 DataFrame 的一行、一列。理解函数接收的数据形状，是使用它的关键。

> 约定：使用 `pd` 作为 Pandas 的别名，使用 `np` 作为 NumPy 的别名。本节可独立从上到下运行。

In [ ]:
import pandas as pd
import numpy as np

## 1. 准备数据路径

本节读取本章 `data` 文件夹中的模拟练习数据，使用 `pathlib` 处理相对路径。

In [ ]:
from pathlib import Path

# 从本章目录运行时使用 data；从项目根目录运行时定位到本章。
data_dir = Path("data")
if not data_dir.is_dir():
    data_dir = Path("第3章-Pandas") / "data"
if not data_dir.is_dir():
    raise FileNotFoundError("未找到本章的 data 文件夹。")

data_dir.as_posix()

In [ ]:
df = pd.read_csv(data_dir / "apply.csv", encoding="utf-8")
df

## 2. 对 Series 的元素应用函数

### 2.1 对成绩开平方

对每个成绩计算平方根，结果仍是 Series。

In [ ]:
df["score"].apply(np.sqrt)

已有 NumPy 向量化函数时，也可以直接作用于整列，通常更简洁。

In [ ]:
np.sqrt(df["score"])

### 2.2 使用 lambda 表达简单规则

本例设定的虚构规则：民族字段不是“汉族”时加 10 分。此处只是条件函数练习，不表示实际考试政策。

In [ ]:
df["other_score"] = df["nation"].apply(lambda value: 0 if value == "汉族" else 10)
df["total_score"] = df["score"] + df["other_score"]
df[["name", "nation", "score", "other_score", "total_score"]]

前两人附加分为 0，第三人附加分为 10。

## 3. 日期转换与间隔

使用 `Series.apply(pd.to_datetime)` 逐个解析日期，可以帮助理解 `apply`；实际处理整列日期时通常直接使用 `pd.to_datetime()`。

In [ ]:
date_from = pd.to_datetime(df["date_from"])
date_to = df["date_to"].apply(pd.to_datetime)
interval = date_to - date_from
interval

相减得到时间间隔 Series。用函数取 `.days` 可以提取天数，也可通过 `.dt.days` 统一提取。

In [ ]:
df["interval"] = interval.apply(lambda value: value.days)
df[["name", "date_from", "date_to", "interval"]]

In [ ]:
interval.dt.days

三人的间隔分别为 9、11、1 天。`.dt.days` 表示间隔中完整的天数；若需要带小数的天数，应使用 `.dt.total_seconds() / 86400`。

## 4. 对 DataFrame 的每一行应用函数

| 写法 | 函数每次接收 |
|---|---|
| `df.apply(func, axis=0)` | 一列，默认行为 |
| `df.apply(func, axis=1)` | 一行 |
| `series.apply(func)` | Series 中的单个元素（本节示例） |

判断规则：原始成绩 `score >= 600` 且间隔 `interval <= 15` 时标记 1，否则标记 0。这里使用原始成绩，不是附加分后的 `total_score`。

In [ ]:
def make_label(row):
    if row["score"] >= 600 and row["interval"] <= 15:
        return 1
    return 0

df["label"] = df.apply(make_label, axis=1)
df[["name", "score", "interval", "label"]]

结果标签为 `1, 1, 0`。函数内 `row` 是一行 Series，因此可以用列名取出多个字段。

## 5. 简单条件优先整列计算

复杂规则适合写命名函数；简单布尔条件可以直接向量化，含义也更紧凑。

In [ ]:
vectorized_label = ((df["score"] >= 600) & (df["interval"] <= 15)).astype(int)
vectorized_label.equals(df["label"])

结果为 `True`，表示本例两种写法得到相同标签。

## 6. 要点总结

- Series 的 `apply` 可用于逐元素转换，DataFrame 的 `apply` 可按行或按列调用函数。
- 涉及多列规则时，使用 `axis=1`，函数通过列名读取这一行的数据。
- 日期先转换为时间类型，再相减得到时间间隔。
- 内置或向量化运算能够表达的简单逻辑，优先直接操作整列。